# Scaling Data Preparation

How do you scale data preparation for real-world ML projects without losing structure or control?

As datasets grow, single-machine Pandas-based workflows begin to break down.

**Pandas limitations — common breaking points:**
- Multi-gigabyte CSV files exceeding RAM
- Memory crashes during large joins
- Single-core processing bottlenecks

The mindset shifts from *"can my laptop handle this?"* to *"how should this run in production at scale?"*

## Topics
1. Chunk-based processing
2. Introduction to Dask
3. Introduction to Polars
4. Automated EDA tools
5. Data validation
6. End-to-end production workflow

In [ ]:
import pandas as pd
import numpy as np
import io

np.random.seed(42)

## 1. Chunk-Based Processing

When datasets exceed memory limits, process in chunks:
1. Read a chunk
2. Transform that chunk
3. Write results or aggregate statistics
4. Repeat until complete

Uses the `chunksize` parameter in `pd.read_csv()`. No distributed system required.

In [ ]:
# Simulate a large CSV in memory (would normally be a file path)
large_data = pd.DataFrame({
    'region':  np.random.choice(['North', 'South', 'East', 'West'], 100_000),
    'revenue': np.random.exponential(5000, 100_000).round(2),
    'cost':    np.random.exponential(2000, 100_000).round(2),
})
csv_buffer = io.StringIO()
large_data.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

# Process in chunks of 10,000 rows
chunk_size  = 10_000
region_totals = {}
rows_processed = 0

for chunk in pd.read_csv(csv_buffer, chunksize=chunk_size):
    chunk['profit'] = chunk['revenue'] - chunk['cost']
    # Aggregate incrementally
    chunk_agg = chunk.groupby('region')['profit'].sum()
    for region, profit in chunk_agg.items():
        region_totals[region] = region_totals.get(region, 0) + profit
    rows_processed += len(chunk)

print(f"Processed {rows_processed:,} rows in chunks of {chunk_size:,}")
print("\nTotal profit by region:")
for region, profit in sorted(region_totals.items()):
    print(f"  {region}: ${profit:,.0f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Re-run chunk loop tracking cumulative profit per chunk for visualisation
csv_viz = io.StringIO(large_data.to_csv(index=False))
regions_order     = ['East', 'North', 'South', 'West']
chunk_cumulative  = {r: [] for r in regions_order}
running_totals    = {r: 0.0 for r in regions_order}
chunk_numbers     = []

for i, chunk in enumerate(pd.read_csv(csv_viz, chunksize=chunk_size), 1):
    chunk['profit'] = chunk['revenue'] - chunk['cost']
    agg = chunk.groupby('region')['profit'].sum()
    for r in regions_order:
        running_totals[r] += agg.get(r, 0)
        chunk_cumulative[r].append(running_totals[r] / 1e6)
    chunk_numbers.append(i)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Chunk-Based Processing — Incremental Aggregation Without Loading All Data',
             fontsize=13, fontweight='bold')

# Panel 1 — Cumulative profit built chunk by chunk (line per region)
region_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']
for region, color in zip(regions_order, region_colors):
    axes[0].plot(chunk_numbers, chunk_cumulative[region],
                 marker='o', markersize=4, lw=2, color=color, label=region)
axes[0].set_xlabel('Chunks processed')
axes[0].set_ylabel('Cumulative Profit (£M)')
axes[0].set_title(f'Profit Accumulates Chunk by Chunk\n({rows_processed:,} rows, {len(chunk_numbers)} chunks of {chunk_size:,})')
axes[0].legend(title='Region', fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Final aggregated profit by region
final_profits = [region_totals[r] / 1e6 for r in regions_order]
bars = axes[1].bar(regions_order, final_profits,
                   color=region_colors, edgecolor='white', linewidth=1.5, alpha=0.88)
for bar, val in zip(bars, final_profits):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.3,
                 f'£{val:.1f}M', ha='center', fontweight='bold')
axes[1].set_ylabel('Total Profit (£M)')
axes[1].set_title('Final Result: Total Profit by Region\n(aggregated incrementally — no RAM overflow)')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 2. Introduction to Dask

Dask provides **parallel and out-of-core computation** while maintaining a Pandas-like API.

| Feature | Description |
|---|---|
| **Parallel Computing** | Splits data across multiple CPU cores or machines |
| **Out-of-Core Processing** | Handles datasets larger than memory by processing in partitions |
| **Minimal Code Changes** | Mirrors much of Pandas syntax |

> Dask bridges the gap between laptop analytics and distributed systems.

In [ ]:
try:
    import dask.dataframe as dd

    # Convert pandas DataFrame to Dask DataFrame
    ddf = dd.from_pandas(large_data, npartitions=4)

    print("Dask DataFrame info:")
    print(f"  Partitions: {ddf.npartitions}")
    print(f"  Columns: {list(ddf.columns)}")

    # Dask uses lazy evaluation — nothing runs until .compute()
    dask_result = (
        ddf
        .assign(profit=ddf['revenue'] - ddf['cost'])
        .groupby('region')['profit']
        .sum()
        .compute()          # trigger actual computation
    )

    print("\nDask result — profit by region:")
    print(dask_result.sort_index())

except ImportError:
    print("Dask not installed. Install with: pip install dask[dataframe]")
    print()
    print("Dask equivalent of pandas operations:")
    print("  pd.read_csv(file)              → dd.read_csv(file, blocksize='64MB')")
    print("  df.groupby(...).agg(...)       → same syntax, lazy")
    print("  result                         → result.compute()")

## 3. Introduction to Polars

Polars is a modern DataFrame engine built in Rust, designed for high performance.

| Advantage | Detail |
|---|---|
| **Speed** | Often significantly faster than Pandas (Rust backend) |
| **Lazy Evaluation** | Queries optimised before execution |
| **Efficient Memory** | Built on Apache Arrow, reducing memory footprint |

In [ ]:
try:
    import polars as pl
    import time

    # Polars eager mode
    pl_df = pl.from_pandas(large_data)

    start = time.time()
    result_polars = (
        pl_df
        .with_columns((pl.col('revenue') - pl.col('cost')).alias('profit'))
        .group_by('region')
        .agg(pl.col('profit').sum())
        .sort('region')
    )
    elapsed_polars = time.time() - start

    # Pandas equivalent for comparison
    start = time.time()
    result_pandas = (
        large_data
        .assign(profit=large_data['revenue'] - large_data['cost'])
        .groupby('region')['profit']
        .sum()
    )
    elapsed_pandas = time.time() - start

    print(f"Polars: {elapsed_polars:.4f}s")
    print(f"Pandas: {elapsed_pandas:.4f}s")
    print("\nPolars result:")
    print(result_polars)

except ImportError:
    print("Polars not installed. Install with: pip install polars")
    print()
    print("Polars syntax example:")
    print("  df.with_columns((pl.col('revenue') - pl.col('cost')).alias('profit'))")
    print("  df.group_by('region').agg(pl.col('profit').sum())")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time

sizes_bm = [10_000, 50_000, 100_000]
labels_bm = [f'{s//1000}k rows' for s in sizes_bm]
pandas_times = []

for size in sizes_bm:
    _d = large_data.head(size).copy()
    t0 = time.time()
    for _ in range(3):
        _d.assign(profit=_d['revenue'] - _d['cost']).groupby('region')['profit'].sum()
    pandas_times.append((time.time() - t0) / 3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Polars vs Pandas — Performance Across Dataset Sizes',
             fontsize=13, fontweight='bold')

try:
    import polars as pl
    polars_times = []
    for size in sizes_bm:
        _pl = pl.from_pandas(large_data.head(size))
        t0 = time.time()
        for _ in range(3):
            (_pl.with_columns((pl.col('revenue') - pl.col('cost')).alias('profit'))
                .group_by('region').agg(pl.col('profit').sum()))
        polars_times.append((time.time() - t0) / 3)

    x, w = np.arange(len(sizes_bm)), 0.35
    axes[0].bar(x - w/2, pandas_times, w, color='#e63946', alpha=0.85, label='Pandas')
    axes[0].bar(x + w/2, polars_times, w, color='#2a9d8f', alpha=0.85, label='Polars')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels_bm)
    axes[0].set_ylabel('Time (seconds) — lower is better')
    axes[0].set_title('Execution Time per Dataset Size')
    axes[0].legend()
    axes[0].spines[['top', 'right']].set_visible(False)

    speedups = [p / q for p, q in zip(pandas_times, polars_times)]
    bars = axes[1].bar(labels_bm, speedups, color='#3d5a80', edgecolor='white', alpha=0.88)
    for bar, sp in zip(bars, speedups):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.02,
                     f'{sp:.1f}×', ha='center', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('Speedup factor (Pandas ÷ Polars)')
    axes[1].set_title('Polars Speedup over Pandas')
    axes[1].spines[['top', 'right']].set_visible(False)

except ImportError:
    bars = axes[0].bar(labels_bm, pandas_times, color='#e63946', edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, pandas_times):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.0005,
                     f'{val:.4f}s', ha='center', fontsize=9)
    axes[0].set_ylabel('Time (seconds)')
    axes[0].set_title('Pandas Execution Time\n(install polars to compare)')
    axes[0].spines[['top', 'right']].set_visible(False)

    axes[1].text(0.5, 0.5,
                 'pip install polars\n\nExpected speedup: 2–10×\n(Rust + Arrow backend)',
                 ha='center', va='center', transform=axes[1].transAxes,
                 fontsize=12,
                 bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.85))
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 4. Automated EDA Tools

Automated EDA tools accelerate early-stage analysis:
- **Rapid Profiling** — instant summaries of distributions, missing values, correlations
- **Auto-Generated Reports** — interactive HTML dashboards

> Automation increases speed but does not replace analytical thinking. Use automation to accelerate exploration, not to replace reasoning.

In [ ]:
# Manual profiling — the essence of what automated tools do
def quick_profile(df: pd.DataFrame) -> pd.DataFrame:
    profile = pd.DataFrame({
        'dtype':       df.dtypes,
        'null_count':  df.isnull().sum(),
        'null_pct':    (df.isnull().mean() * 100).round(2),
        'unique':      df.nunique(),
        'mean':        df.mean(numeric_only=True),
        'std':         df.std(numeric_only=True),
        'skew':        df.skew(numeric_only=True),
    })
    return profile


print("Dataset profile:")
quick_profile(large_data)

In [ ]:
import matplotlib.pyplot as plt

profile = quick_profile(large_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Profile Dashboard — Replicating What Auto EDA Tools Surface',
             fontsize=13, fontweight='bold')

# Panel 1 — Null % per column
null_pct = profile['null_pct'].fillna(0).sort_values(ascending=False)
col_null_colors = ['#e63946' if v > 0 else '#2a9d8f' for v in null_pct.values]
axes[0].barh(null_pct.index, null_pct.values,
             color=col_null_colors, edgecolor='white', alpha=0.85)
axes[0].axvline(5, color='#555', ls='--', lw=1.2, alpha=0.7, label='5% alert threshold')
axes[0].set_xlabel('Missing %')
axes[0].set_title('Null % per Column\n(red = has missing data)')
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Skewness for numeric columns
skew_data = profile['skew'].dropna().sort_values()
skew_colors = ['#e63946' if abs(v) > 1 else '#2a9d8f' for v in skew_data.values]
axes[1].barh(skew_data.index, skew_data.values, color=skew_colors, edgecolor='white', alpha=0.85)
axes[1].axvline( 1, color='#555', ls='--', lw=1.2, alpha=0.6, label='|skew| = 1')
axes[1].axvline(-1, color='#555', ls='--', lw=1.2, alpha=0.6)
axes[1].axvline(0,  color='#000', lw=0.8, alpha=0.3)
axes[1].set_xlabel('Skewness')
axes[1].set_title('Skewness per Column\n(|skew| > 1 → consider log transform)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Revenue distribution (the most skewed column)
axes[2].hist(large_data['revenue'], bins=60, color='#FFB74D', edgecolor='white', alpha=0.85)
axes[2].axvline(large_data['revenue'].mean(),   color='#e63946', lw=2, ls='--',
                label=f'Mean   = {large_data["revenue"].mean():,.0f}')
axes[2].axvline(large_data['revenue'].median(), color='#2a9d8f', lw=2, ls='--',
                label=f'Median = {large_data["revenue"].median():,.0f}')
axes[2].set_xlabel('Revenue')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Revenue Distribution\nskew = {large_data["revenue"].skew():.2f}  (right-skewed)')
axes[2].legend(fontsize=8)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ydata-profiling (formerly pandas-profiling) generates a full HTML report
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(large_data.head(1000), title='Dataset Profile', minimal=True)
    profile.to_file('/tmp/eda_report.html')
    print("Report saved to /tmp/eda_report.html")
except ImportError:
    print("Install ydata-profiling with: pip install ydata-profiling")
    print("Then use: ProfileReport(df).to_file('report.html')")

## 5. Data Validation

Data pipelines must enforce quality standards. Silent data drift is one of the biggest production risks.

**Validation frameworks** (Great Expectations, Pandera) let you define:
- **Schema Validation** — column names, types, ranges, constraints
- **Data Quality Checks** — null %, duplicates, outlier detection, distribution shifts

In [ ]:
# Lightweight manual validation — same principle as Great Expectations
def validate_schema(df: pd.DataFrame, rules: dict) -> list:
    failures = []

    for col, constraints in rules.items():
        if col not in df.columns:
            failures.append(f"MISSING COLUMN: {col}")
            continue

        null_pct = df[col].isnull().mean()
        if null_pct > constraints.get('max_null_pct', 1.0):
            failures.append(f"{col}: null_pct={null_pct:.2%} > {constraints['max_null_pct']:.0%}")

        if 'min' in constraints and df[col].min() < constraints['min']:
            failures.append(f"{col}: min={df[col].min()} < {constraints['min']}")

        if 'max' in constraints and df[col].max() > constraints['max']:
            failures.append(f"{col}: max={df[col].max()} > {constraints['max']}")

    return failures


schema_rules = {
    'revenue':  {'max_null_pct': 0.01, 'min': 0},
    'cost':     {'max_null_pct': 0.01, 'min': 0},
    'region':   {'max_null_pct': 0.05},
}

failures = validate_schema(large_data, schema_rules)
if failures:
    print("VALIDATION FAILURES:")
    for f in failures:
        print(f"  - {f}")
else:
    print("All validation checks passed.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Evaluate each rule with its actual vs allowed value
rules_detail = [
    ('revenue\nnull% ≤ 1%',  large_data['revenue'].isnull().mean() * 100, 1.0),
    ('cost\nnull% ≤ 1%',     large_data['cost'].isnull().mean()    * 100, 1.0),
    ('region\nnull% ≤ 5%',   large_data['region'].isnull().mean()  * 100, 5.0),
    ('revenue\nmin ≥ 0',      large_data['revenue'].min(),                0.0),
    ('cost\nmin ≥ 0',         large_data['cost'].min(),                   0.0),
]
passed = [actual <= threshold for _, actual, threshold in rules_detail]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Data Validation — Rule-Based Quality Gates',
             fontsize=13, fontweight='bold')

# Panel 1 — Traffic-light pass/fail per rule
labels = [r for r, _, _ in rules_detail]
colors_pf = ['#2a9d8f' if p else '#e63946' for p in passed]
bars = axes[0].barh(labels, [1] * len(rules_detail),
                    color=colors_pf, edgecolor='white', alpha=0.88, height=0.5)
for bar, p in zip(bars, passed):
    axes[0].text(0.5, bar.get_y() + bar.get_height() / 2,
                 'PASS' if p else 'FAIL',
                 ha='center', va='center',
                 fontweight='bold', fontsize=12, color='white')
axes[0].set_xlim(0, 1)
axes[0].set_xticks([])
axes[0].set_title(f'Validation Results — {sum(passed)}/{len(passed)} rules passed')
axes[0].spines[['top', 'right', 'bottom']].set_visible(False)

# Panel 2 — Null % actual vs max allowed (for the null-% rules)
null_rules  = rules_detail[:3]
null_labels = [r for r, _, _ in null_rules]
actuals     = [a for _, a, _ in null_rules]
thresholds_ = [t for _, _, t in null_rules]

x, w = np.arange(len(null_rules)), 0.32
axes[1].bar(x - w/2, actuals,     w, color='#3d5a80', alpha=0.85, label='Actual %')
axes[1].bar(x + w/2, thresholds_, w, color='#e63946', alpha=0.5,  label='Max allowed %')
axes[1].set_xticks(x)
axes[1].set_xticklabels(null_labels, fontsize=9)
axes[1].set_ylabel('Missing %')
axes[1].set_title('Null % — Actual vs Threshold\n(actual must stay below threshold)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Pandera: schema-based validation (more concise and declarative)
try:
    import pandera as pa

    schema = pa.DataFrameSchema({
        'revenue': pa.Column(float, pa.Check.greater_than_or_equal_to(0)),
        'cost':    pa.Column(float, pa.Check.greater_than_or_equal_to(0)),
        'region':  pa.Column(str,   pa.Check.isin(['North', 'South', 'East', 'West']),
                             nullable=True),
    })

    validated = schema.validate(large_data)
    print(f"Pandera validation passed: {validated.shape[0]:,} rows")

except ImportError:
    print("Install Pandera with: pip install pandera")

## 6. End-to-End Production Workflow

The complete structured sequence:

```
Load → Clean → Engineer → Validate → Model
```

Every successful production system follows this lifecycle. Skipping steps results in unstable systems.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)


def load_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    logger.info(f"[LOAD] {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
    return df_raw.copy()


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    before = df.isnull().sum().sum()
    df = df.fillna({'revenue': df['revenue'].median(),
                    'cost': df['cost'].median(),
                    'region': 'Unknown'})
    after = df.isnull().sum().sum()
    logger.info(f"[CLEAN] Nulls reduced: {before} → {after}")
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df['profit']         = df['revenue'] - df['cost']
    df['margin_pct']     = (df['profit'] / df['revenue']).round(4)
    df['log_revenue']    = np.log1p(df['revenue'])
    logger.info(f"[ENGINEER] Features: {list(df.columns)}")
    return df


def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    assert df.isnull().sum().sum() == 0,     "Nulls remain after cleaning"
    assert (df['revenue'] >= 0).all(),        "Negative revenue detected"
    assert (df['cost']    >= 0).all(),        "Negative cost detected"
    logger.info(f"[VALIDATE] All checks passed for {df.shape[0]:,} rows")
    return df


# Run the production workflow
pipeline_output = (
    large_data
    .pipe(load_data)
    .pipe(clean_data)
    .pipe(engineer_features)
    .pipe(validate_data)
)

print(f"\nFinal dataset: {pipeline_output.shape}")
pipeline_output.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('End-to-End Pipeline — Effect of Each Stage on the Data',
             fontsize=13, fontweight='bold')

# Panel 1 — Column count per stage
stage_labels = ['Load', 'Clean', 'Engineer\n+ Validate']
col_counts   = [large_data.shape[1], large_data.shape[1], pipeline_output.shape[1]]
colors_s     = ['#1565C0', '#2E7D32', '#6A1B9A']
bars = axes[0].bar(stage_labels, col_counts, color=colors_s,
                   edgecolor='white', linewidth=1.5, alpha=0.88)
for bar, cnt in zip(bars, col_counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.05, str(cnt),
                 ha='center', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of columns')
axes[0].set_title('Feature Count per Stage\n(Engineer adds profit, margin_pct, log_revenue)')
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Revenue: raw distribution vs engineered log_revenue
axes[1].hist(large_data['revenue'], bins=60, color='#e63946',
             alpha=0.5, density=True,
             label=f'Raw  (skew={large_data["revenue"].skew():.2f})')
axes[1].hist(pipeline_output['log_revenue'], bins=60, color='#2a9d8f',
             alpha=0.5, density=True,
             label=f'log_revenue  (skew={pipeline_output["log_revenue"].skew():.2f})')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density')
axes[1].set_title('Revenue: Raw vs log1p\n(Engineer stage — skewness reduced)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Profit margin by region (newly engineered feature)
for region in sorted(pipeline_output['region'].dropna().unique()):
    subset = pipeline_output[pipeline_output['region'] == region]['margin_pct']
    axes[2].hist(subset, bins=30, alpha=0.45, density=True, label=region)
axes[2].set_xlabel('Margin %')
axes[2].set_ylabel('Density')
axes[2].set_title('Profit Margin by Region\n(Engineered feature: (revenue-cost)/revenue)')
axes[2].legend(title='Region', fontsize=8)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Common Mistakes and Best Practices

| Mistake | Best Practice |
|---|---|
| **Overusing `.apply()`** | Prefer vectorised operations |
| **Hard-coding logic** | Move thresholds to config files |
| **Ignoring memory constraints** | Monitor DataFrame size; use dtypes optimisation |
| **Data leakage** | Isolate training/testing transformations |

Professional workflows adhere to consistent standards:
- **Modular code** — break large pipelines into reusable functions and classes
- **Clear naming** — descriptive variable and function names
- **Feature definitions** — document transformation logic and feature intent
- **Reproducibility** — set random seeds, track dependencies, version control

In [ ]:
# Memory monitoring — track DataFrame size across pipeline stages
def log_memory(df: pd.DataFrame, stage: str):
    mb = df.memory_usage(deep=True).sum() / 1024**2
    logger.info(f"[{stage}] Memory: {mb:.2f} MB | Shape: {df.shape}")


log_memory(large_data,      'INPUT')
log_memory(pipeline_output, 'OUTPUT')

In [ ]:
import matplotlib.pyplot as plt

# Memory per stage
stage_names = ['Input\n(large_data)', 'Pipeline\nOutput']
stage_dfs   = [large_data, pipeline_output]
stage_mb    = [df_.memory_usage(deep=True).sum() / 1024**2 for df_ in stage_dfs]
stage_cols_  = ['#1565C0', '#6A1B9A']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Memory Monitoring — Footprint Across Pipeline Stages',
             fontsize=13, fontweight='bold')

# Panel 1 — Memory per stage
bars = axes[0].bar(stage_names, stage_mb, color=stage_cols_,
                   edgecolor='white', linewidth=1.5, alpha=0.88, width=0.4)
for bar, val in zip(bars, stage_mb):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02,
                 f'{val:.2f} MB', ha='center', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Memory (MB)')
axes[0].set_title('Total Memory per Stage\n(adding features increases footprint)')
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Memory per column in the pipeline output
col_mem_kb = pipeline_output.memory_usage(deep=True)[1:] / 1024
col_mem_sorted = col_mem_kb.sort_values(ascending=False)
mean_kb = col_mem_kb.mean()
col_colors = ['#e63946' if v > mean_kb else '#2a9d8f' for v in col_mem_sorted.values]
axes[1].barh(col_mem_sorted.index, col_mem_sorted.values,
             color=col_colors, edgecolor='white', alpha=0.85)
axes[1].axvline(mean_kb, color='#555', lw=1.5, ls='--',
                label=f'Mean = {mean_kb:.1f} KB')
axes[1].set_xlabel('Memory (KB)')
axes[1].set_title('Memory per Column (Output)\n(red = above average)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## Key Takeaways

- **Chunk-based processing** enables large-scale workflows without distributed systems — practical when infrastructure is limited.
- **Dask** provides a Pandas-like API for parallel and out-of-core computation.
- **Polars** offers superior performance via Rust and lazy evaluation.
- **Data validation** transforms assumptions into enforceable rules — prevents silent failures.
- The complete production sequence is: **Load → Clean → Engineer → Validate → Model**.
- Avoid common mistakes: `.apply()` overuse, hard-coded values, ignoring memory, data leakage.